In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2009_2010"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "HIQ_F.xpt"
csv_file = DATA_DIR / "HIQ_F_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve untouched XPT values.
#
source_df = df.copy()


print("=" * 75)
print("NHANES 2009-2010 HIQ_F XPT -> CSV")
print("=" * 75)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "HIQ011",
    "HIQ031A",
    "HIQ031B",
    "HIQ031C",
    "HIQ031D",
    "HIQ031E",
    "HIQ031F",
    "HIQ031H",
    "HIQ031I",
    "HIQ031J",
    "HIQ031AA",
    "HIQ260",
    "HIQ105",
    "HIQ270",
    "HIQ210"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "HIQ_F column names/order differ from expected.\n"
        f"Actual: {df.columns.tolist()}"
    )


if len(df) != 10537:

    raise ValueError(
        f"Unexpected row count: {len(df)}"
    )


if len(df.columns) != 16:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 10,537 rows x 16 columns."
)


# ============================================================
# 4. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas/SAS XPORT zero-decoding artifact:
#
#     5.397605346934028e-79
#
# The attached HIQ_F contains NONE.
#
# Do not automatically replace anything if a future
# version unexpectedly contains this value.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = df.select_dtypes(
    include=[np.number]
).columns


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_total = int(
    df[numeric_cols]
    .eq(0)
    .sum()
    .sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")


print(
    "True numeric zeros:",
    f"{true_zero_total:,}"
)


print(
    "Tiny artifact occurrences:",
    f"{tiny_total:,}"
)


if tiny_total > 0:

    print(
        "\nTiny values by variable:"
    )

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )

    raise ValueError(
        "STOP: Unexpected tiny XPORT artifacts "
        "found in HIQ_F. "
        "Do not automatically replace."
    )


print(
    "PASS: No tiny XPORT zero-decoding artifacts."
)


# ============================================================
# 5. CHECK FOR GENUINE DECIMAL VALUES
# ============================================================

#
# CRITICAL FIDELITY RULE:
#
# Never round genuine decimal values.
#
# This test occurs BEFORE Int64 conversion.
#
# HIQ_F contains no genuine fractional variables.
#
# This same protection remains important for:
#
#     Dietary
#     Laboratory
#     Examination
#     Sample weights
#     Continuous measurements
#

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_columns = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    if fractional_mask.any():

        fractional_columns.append(
            col
        )

        print(
            f"{col}: "
            f"{int(fractional_mask.sum()):,} "
            "fractional values"
        )


if fractional_columns:

    raise ValueError(
        "STOP: Genuine decimal values detected. "
        "They will NOT be rounded:\n"
        + str(fractional_columns)
    )


print(
    "PASS: No genuine fractional values "
    "exist in HIQ_F."
)


# ============================================================
# 6. CLEAN AND VALIDATE SEQN
# ============================================================

seqn = (
    df["SEQN"]
    .dropna()
)


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=1e-12
    )
):

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")


print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)


print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


print(
    "Minimum:",
    df["SEQN"].min()
)


print(
    "Maximum:",
    df["SEQN"].max()
)


if (
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
    != 0
    or
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
    != 0
    or
    df["SEQN"].min()
    != 51624
    or
    df["SEQN"].max()
    != 62160
):

    raise ValueError(
        "SEQN validation failed."
    )


print(
    "PASS: SEQN validated."
)


# ============================================================
# 7. FORMAT QUESTIONNAIRE VALUES WITHOUT .0
# ============================================================

#
# All HIQ_F fields contain whole-number questionnaire
# codes after the fractional-value guard above.
#
# NO .round() is used.
#

for col in df.columns:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) > 0:

        whole_number_check = np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )


        if not np.all(
            whole_number_check
        ):

            raise ValueError(
                f"{col} contains genuine decimal values "
                "and cannot safely be converted to Int64."
            )


    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: All HIQ_F fields formatted "
    "without unnecessary .0."
)


# ============================================================
# 8. CDC CHECK - HIQ011
# ============================================================

#
# Covered by health insurance
#
# CDC:
#
# 1 = Yes          8,500
# 2 = No           2,022
# 7 = Refused          4
# 9 = Don't know      10
# Missing              1
#

EXPECTED_HIQ011 = {
    1: 8500,
    2: 2022,
    7: 4,
    9: 10
}


actual = (
    df["HIQ011"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if (
    actual != EXPECTED_HIQ011
    or
    int(
        df["HIQ011"]
        .isna()
        .sum()
    )
    != 1
):

    raise ValueError(
        "HIQ011 differs from CDC."
    )


print(
    "PASS: HIQ011 matches CDC."
)


# ============================================================
# 9. CDC CHECK - INSURANCE TYPE VARIABLES
# ============================================================

#
# These are essentially "code-all-that-apply" style
# insurance coverage indicators.
#
# Non-selected insurance types remain missing.
#
# CDC frequencies:
#
# HIQ031A 14 = Private insurance            4,894
#          99 = Don't know                     24
#
# HIQ031B 15 = Medicare                     1,461
#
# HIQ031C 16 = Medi-Gap                        24
#
# HIQ031D 17 = Medicaid                     1,828
#
# HIQ031E 18 = SCHIP                           92
#
# HIQ031F 19 = Military health care           321
#
# HIQ031H 21 = State-sponsored health plan    738
#
# HIQ031I 22 = Other government insurance     230
#
# HIQ031J 23 = Single service plan            295
#
# HIQ031AA 40 = No coverage of any type         0
#

CDC_INSURANCE_TYPES = {

    "HIQ031A": {
        "values": {
            14: 4894,
            99: 24
        },
        "missing": 5619
    },

    "HIQ031B": {
        "values": {
            15: 1461
        },
        "missing": 9076
    },

    "HIQ031C": {
        "values": {
            16: 24
        },
        "missing": 10513
    },

    "HIQ031D": {
        "values": {
            17: 1828
        },
        "missing": 8709
    },

    "HIQ031E": {
        "values": {
            18: 92
        },
        "missing": 10445
    },

    "HIQ031F": {
        "values": {
            19: 321
        },
        "missing": 10216
    },

    "HIQ031H": {
        "values": {
            21: 738
        },
        "missing": 9799
    },

    "HIQ031I": {
        "values": {
            22: 230
        },
        "missing": 10307
    },

    "HIQ031J": {
        "values": {
            23: 295
        },
        "missing": 10242
    },

    "HIQ031AA": {
        "values": {},
        "missing": 10537
    }
}


print("\n--- CDC INSURANCE TYPE CHECKS ---")


for col, expected in CDC_INSURANCE_TYPES.items():

    actual_values = (
        df[col]
        .value_counts()
        .sort_index()
        .to_dict()
    )


    actual_missing = int(
        df[col]
        .isna()
        .sum()
    )


    if actual_values != expected["values"]:

        raise ValueError(
            f"{col} frequencies differ from CDC.\n"
            f"Expected: {expected['values']}\n"
            f"Actual:   {actual_values}"
        )


    if actual_missing != expected["missing"]:

        raise ValueError(
            f"{col} missing count differs from CDC.\n"
            f"Expected: {expected['missing']}\n"
            f"Actual:   {actual_missing}"
        )


    print(
        f"PASS: {col} matches CDC."
    )


# ============================================================
# 10. CDC CHECK - HIQ260
# ============================================================

#
# Have Medicare?
#
# CDC:
#
# 1 = Yes            129
# 2 = No             124
# 7 = Refused          0
# 9 = Don't know       4
# Missing         10,280
#

EXPECTED_HIQ260 = {
    1: 129,
    2: 124,
    9: 4
}


actual = (
    df["HIQ260"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if (
    actual != EXPECTED_HIQ260
    or
    int(
        df["HIQ260"]
        .isna()
        .sum()
    )
    != 10280
):

    raise ValueError(
        "HIQ260 differs from CDC."
    )


print(
    "PASS: HIQ260 matches CDC."
)


# ============================================================
# 11. CDC CHECK - HIQ105
# ============================================================

#
# Insurance card available or not
#
# CDC:
#
# 1 = Yes       950
# 2 = No        113
# Missing     9,474
#

EXPECTED_HIQ105 = {
    1: 950,
    2: 113
}


actual = (
    df["HIQ105"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if (
    actual != EXPECTED_HIQ105
    or
    int(
        df["HIQ105"]
        .isna()
        .sum()
    )
    != 9474
):

    raise ValueError(
        "HIQ105 differs from CDC."
    )


print(
    "PASS: HIQ105 matches CDC."
)


# ============================================================
# 12. CDC CHECK - HIQ270
# ============================================================

#
# Do plans cover prescriptions?
#
# CDC:
#
# 1 = Yes          7,984
# 2 = No             458
# 7 = Refused          8
# 9 = Don't know      59
# Missing          2,028
#

EXPECTED_HIQ270 = {
    1: 7984,
    2: 458,
    7: 8,
    9: 59
}


actual = (
    df["HIQ270"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if (
    actual != EXPECTED_HIQ270
    or
    int(
        df["HIQ270"]
        .isna()
        .sum()
    )
    != 2028
):

    raise ValueError(
        "HIQ270 differs from CDC."
    )


print(
    "PASS: HIQ270 matches CDC."
)


# ============================================================
# 13. CDC CHECK - HIQ210
# ============================================================

#
# Any time without health insurance in past year?
#
# CDC:
#
# 1 = Yes            510
# 2 = No           7,996
# 7 = Refused          0
# 9 = Don't know       3
# Missing          2,028
#

EXPECTED_HIQ210 = {
    1: 510,
    2: 7996,
    9: 3
}


actual = (
    df["HIQ210"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if (
    actual != EXPECTED_HIQ210
    or
    int(
        df["HIQ210"]
        .isna()
        .sum()
    )
    != 2028
):

    raise ValueError(
        "HIQ210 differs from CDC."
    )


print(
    "PASS: HIQ210 matches CDC."
)


# ============================================================
# 14. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# HIQ_F requires NO numeric correction.
#
# Therefore every numeric value should remain exactly
# equal to the source XPT.
#
# Int64 conversion changes only representation:
#
#     51624.0 -> 51624
#     1.0     -> 1
#     14.0    -> 14
#

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


failures = []


for col in source_df.columns:

    expected_values = (
        source_df[col]
        .to_numpy()
    )


    current_values = (
        df[col]
        .astype("Float64")
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        failures.append(
            col
        )


if failures:

    raise ValueError(
        "Unexpected value changes before export:\n"
        + str(failures)
    )


print(
    "PASS: All 16 variables preserve "
    "original XPT values exactly."
)


# ============================================================
# 15. EXPORT CSV
# ============================================================

#
# IMPORTANT:
#
# NO float_format.
# NO rounding.
#
# Missing values become blank.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")

print(
    csv_file
)


# ============================================================
# 16. RE-READ EXPORTED CSV
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 17. EXACT XPT -> CSV FIDELITY
# ============================================================

print("\n--- XPT -> CSV FIDELITY ---")


csv_failures = []


for col in source_df.columns:

    expected_values = (
        source_df[col]
        .to_numpy()
    )


    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "XPT -> CSV differences found in:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 16 CSV columns match "
    "source XPT values exactly."
)


# ============================================================
# 18. MISSING VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in source_df.columns:

    expected_missing = (
        source_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print("\n--- MISSING VALUE CHECK ---")


print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing values preserved exactly."
)


# ============================================================
# 19. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in csv_df.select_dtypes(
    include=[np.number]
).columns:

    tiny_csv_count += int(
        csv_df[col]
        .eq(TINY_VALUE)
        .sum()
    )


print("\n--- FINAL TINY VALUE CHECK ---")


print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Unexpected tiny XPORT artifacts found "
        "in final CSV."
    )


print(
    "PASS: No tiny-value artifacts."
)


# ============================================================
# 20. RAW CSV .0 FORMAT CHECK
# ============================================================

#
# HIQ_F contains NO genuine decimal fields.
#
# Therefore ".0" should not appear anywhere in
# this particular CSV.
#

print("\n--- RAW CSV FORMAT CHECK ---")


with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


dot_zero_count = (
    csv_text.count(".0")
)


print(
    "Occurrences of '.0':",
    dot_zero_count
)


if dot_zero_count != 0:

    raise ValueError(
        "Unexpected .0 formatting remains."
    )


print(
    "PASS: No unnecessary .0 formatting."
)


# ============================================================
# 21. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = (
    csv_text.lower().count("e-")
    +
    csv_text.lower().count("e+")
)


print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 22. FINAL CSV SEQN CHECK
# ============================================================

print("\n--- FINAL CSV SEQN CHECK ---")


if (
    csv_df["SEQN"].min()
    != 51624
    or
    csv_df["SEQN"].max()
    != 62160
    or
    int(
        csv_df["SEQN"]
        .isna()
        .sum()
    )
    != 0
    or
    int(
        csv_df["SEQN"]
        .duplicated()
        .sum()
    )
    != 0
):

    raise ValueError(
        "Final CSV SEQN validation failed."
    )


print(
    "PASS: Final CSV SEQN is valid."
)


# ============================================================
# 23. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 75
)


print(
    "HIQ_F CONVERSION COMPLETE"
)


print(
    "=" * 75
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "Original tiny-value artifacts:",
    tiny_total
)


print(
    "Tiny-value corrections: NONE"
)


print(
    "Genuine decimal variables: NONE"
)


print(
    "XPT -> CSV unexpected differences: 0 expected"
)


print(
    "Missing-value differences: 0 expected"
)


print(
    "Tiny values remaining: 0 expected"
)


print(
    "Unnecessary .0: 0 expected"
)


print(
    "Scientific notation: 0 expected"
)

NHANES 2009-2010 HIQ_F XPT -> CSV
Rows: 10,537
Columns: 16
PASS: Structure = 10,537 rows x 16 columns.

--- ORIGINAL XPT ZERO CHECK ---
True numeric zeros: 0
Tiny artifact occurrences: 0
PASS: No tiny XPORT zero-decoding artifacts.

--- FRACTIONAL VALUE CHECK ---
PASS: No genuine fractional values exist in HIQ_F.

--- SEQN ---
Missing: 0
Duplicates: 0
Minimum: 51624
Maximum: 62160
PASS: SEQN validated.

PASS: All HIQ_F fields formatted without unnecessary .0.
PASS: HIQ011 matches CDC.

--- CDC INSURANCE TYPE CHECKS ---
PASS: HIQ031A matches CDC.
PASS: HIQ031B matches CDC.
PASS: HIQ031C matches CDC.
PASS: HIQ031D matches CDC.
PASS: HIQ031E matches CDC.
PASS: HIQ031F matches CDC.
PASS: HIQ031H matches CDC.
PASS: HIQ031I matches CDC.
PASS: HIQ031J matches CDC.
PASS: HIQ031AA matches CDC.
PASS: HIQ260 matches CDC.
PASS: HIQ105 matches CDC.
PASS: HIQ270 matches CDC.
PASS: HIQ210 matches CDC.

--- VALUE FIDELITY BEFORE EXPORT ---
PASS: All 16 variables preserve original XPT values exactly.

